# Data Augmentation at Scale — BioCatch Vishing Dataset (1M rows) — v2

## Cambios frente a v1

1. **Copula gaussiano mixto sobre TODAS las features numéricas** (continuas + integer + binarias). Antes solo las continuas estaban correlacionadas entre sí; las binarias e integer se muestreaban independientes, lo que destruía la co-ocurrencia `phone_call_active` + `segmented_typing_ratio` + `hesitation_count` que constituye la huella de vishing.
2. **Correlación estimada sobre normal scores** (no Pearson sobre los datos crudos). Esto convierte el approach en un copula gaussiano formalmente correcto y respeta las marginales sesgadas.
3. **Features derivadas se recalculan, no se muestrean** (`errors_per_minute`, `interactions_per_s`, `hesitation_composite`).
4. **Consistencias lógicas forzadas**: `is_atypical_hour = f(hour_of_day)` determinístico, `biocatch_*_indicator` condicionados empíricamente en el `biocatch_risk_score` sintético.
5. **Filtrado a `device_type='mobile'`** — se descartan sesiones web (las columnas de touch/gyro no aplican).
6. **Marginales continuas por interpolación lineal** entre cuantiles empíricos (elimina el efecto de escalones del `int(u*n)` original).
7. **`days_to_claim` bootstrap** de la distribución empírica original en lugar de `exponential(3)`.
8. **Notebook de validación integrado**: KS por feature, distancia Frobenius de matrices de correlación y test de distinguibilidad con XGBoost.

## Configuración objetivo

- Total: 1,000,000 sesiones mobile
- `is_vishing`=1: ~1.5%
- Todas las sesiones mobile originales se incluyen intactas y marcadas con `is_synthetic=0`.


## 1. Setup y carga

In [ ]:
import boto3
import sagemaker

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
default_bucket = sagemaker_session.default_bucket()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
import warnings
warnings.filterwarnings('ignore')
from scipy import stats as scipy_stats
import pyarrow

plt.rcParams.update({
    'figure.figsize': (14, 6), 'font.size': 11, 'axes.titlesize': 14,
    'axes.labelsize': 12, 'figure.dpi': 100,
    'axes.spines.top': False, 'axes.spines.right': False,
})

COLORS = {'legit': '#2ecc71', 'vishing': '#e74c3c', 'neutral': '#3498db'}
RNG = np.random.default_rng(42)

print('✅ Libraries loaded')

In [ ]:
s3_input_path = 's3://poc-fraude-vishing/biocatch_sinthetic_data.csv'
df_original = pd.read_csv(s3_input_path, parse_dates=['session_timestamp'])

print(f'Original dataset: {len(df_original):,} rows × {df_original.shape[1]} columns')
print(f'  Legitimate: {(df_original["is_vishing"]==0).sum():,}')
print(f'  Vishing:    {(df_original["is_vishing"]==1).sum():,} ({df_original["is_vishing"].mean()*100:.1f}%)')

## 2. Filtrado a `device_type='mobile'`

Descartamos las sesiones web porque las columnas de touch dynamics, gyro y acelerómetro no son físicamente coherentes en desktop. Todo el aumento se hace únicamente sobre el subconjunto mobile.

In [ ]:
df_mobile = df_original[df_original['device_type'] == 'mobile'].copy().reset_index(drop=True)

print(f'Mobile sessions: {len(df_mobile):,} ({len(df_mobile)/len(df_original)*100:.1f}% del original)')
print(f'  Legitimate mobile: {(df_mobile["is_vishing"]==0).sum():,}')
print(f'  Vishing mobile:    {(df_mobile["is_vishing"]==1).sum():,} ({df_mobile["is_vishing"].mean()*100:.2f}%)')
print(f'  OS breakdown: {df_mobile["os_type"].value_counts().to_dict()}')

## 3. Clasificación de features

**Cambios clave frente a v1**:

- Las features **derivadas** (`errors_per_minute`, `interactions_per_s`, `hesitation_composite`) salen de `continuous_cols` porque las recalculamos al final desde sus componentes.
- `is_atypical_hour` sale de `binary_cols` porque es función determinística de `hour_of_day`.
- `biocatch_*_indicator` se manejan aparte con condicional empírica sobre `biocatch_risk_score`.

In [ ]:
# Metadata (se regenera al final)
meta_cols = ['session_id', 'customer_id', 'session_timestamp',
             'device_type', 'os_type', 'app_version']

label_cols = ['is_vishing', 'days_to_claim', 'claim_category']

# Continuas que SÍ se muestrean con el copula
continuous_cols = [
    'avg_keyhold_ms', 'avg_interkey_latency_ms', 'typing_speed_cps',
    'keystroke_variability', 'segmented_typing_ratio',
    'avg_touch_pressure', 'avg_touch_size_px', 'swipe_speed_px_s',
    'swipe_directional_variance', 'scroll_speed_avg',
    'device_tilt_angle_mean', 'device_tilt_variability',
    'gyro_rotation_rate_mean', 'accelerometer_jerk_mean',
    'avg_hesitation_duration_s', 'max_hesitation_duration_s',
    'total_dead_time_s', 'dead_time_ratio',
    'screen_transition_time_avg_s', 'data_familiarity_score',
    'session_duration_s', 'call_overlap_duration_s',
    'time_to_transaction_s',
]

# DERIVADAS -> se recalculan, no se muestrean
derived_cols = ['errors_per_minute', 'interactions_per_s', 'hesitation_composite']

# Integer con la CDF empírica
integer_cols = [
    'phone_motion_events', 'hesitation_count', 'dead_time_periods',
    'screens_visited', 'unique_screens_visited', 'unusual_screen_visits',
    'navigation_back_count', 'input_error_count', 'input_correction_count',
    'amount_field_corrections', 'beneficiary_field_corrections',
    'copy_paste_events', 'doodling_events', 'hour_of_day',
    'transaction_amount_cop',
    'biocatch_risk_score', 'biocatch_genuine_score',
]

# Binarias que sí van en el copula
binary_cols = [
    'phone_call_active',
    'remote_access_tool_detected', 'suspicious_app_detected',
    'transaction_attempted', 'is_new_beneficiary',
]

# Binarias que se derivan por reglas o condicional empírica (fuera del copula)
rule_binary_cols = ['is_atypical_hour']
conditional_binary_cols = ['biocatch_ato_indicator', 'biocatch_social_eng_indicator', 'biocatch_bot_indicator']

all_copula_cols = continuous_cols + integer_cols + binary_cols

print(f'Continuas en copula:  {len(continuous_cols)}')
print(f'Integer en copula:    {len(integer_cols)}')
print(f'Binarias en copula:   {len(binary_cols)}')
print(f'Total en copula:      {len(all_copula_cols)}')
print(f'Derivadas (recalc):   {len(derived_cols)}')
print(f'Binarias por regla:   {len(rule_binary_cols)}')
print(f'Binarias condicional: {len(conditional_binary_cols)}')

## 4. Helpers de reglas y condicionales empíricas

Precalculamos:

- La distribución empírica de `days_to_claim` en la clase vishing original.
- Las probabilidades condicionales `P(indicator=1 | risk_score_bin)` para cada indicador BioCatch, por clase.


In [ ]:
ATYPICAL_HOURS = {22, 23, 0, 1, 2, 3, 4, 5}

def compute_atypical_hour(hour_series):
    return hour_series.isin(ATYPICAL_HOURS).astype(int)

# Bootstrap pool para days_to_claim (solo vishing original)
days_to_claim_pool = df_mobile.loc[df_mobile['is_vishing']==1, 'days_to_claim'].values
print(f'days_to_claim pool size: {len(days_to_claim_pool)} (rango {days_to_claim_pool.min()}–{days_to_claim_pool.max()})')

# Distribución empírica de claim_category en originales vishing
claim_cat_probs = df_mobile.loc[df_mobile['is_vishing']==1, 'claim_category'].value_counts(normalize=True)
print(f'claim_category distribution:\n{claim_cat_probs}')

In [ ]:
def build_conditional_indicator_table(df_class, risk_col='biocatch_risk_score',
                                       indicator_cols=('biocatch_ato_indicator',
                                                       'biocatch_social_eng_indicator',
                                                       'biocatch_bot_indicator'),
                                       n_bins=10):
    """
    Devuelve, para cada indicador, un vector de probabilidad condicional
    P(indicator=1 | risk_score in bin_k) y los bordes de los bins.
    """
    edges = np.quantile(df_class[risk_col], np.linspace(0, 1, n_bins + 1))
    edges[0] -= 1e-6
    edges[-1] += 1e-6
    df_class = df_class.copy()
    df_class['_bin'] = pd.cut(df_class[risk_col], bins=edges, labels=False, include_lowest=True)
    table = {}
    for col in indicator_cols:
        probs = df_class.groupby('_bin')[col].mean().reindex(range(n_bins), fill_value=df_class[col].mean()).values
        table[col] = probs
    return edges, table

edges_legit, cond_table_legit = build_conditional_indicator_table(df_mobile[df_mobile['is_vishing']==0])
edges_vish,  cond_table_vish  = build_conditional_indicator_table(df_mobile[df_mobile['is_vishing']==1])

print('Condicional legit (P(indicator=1 | risk_score bin)):')
for k, v in cond_table_legit.items():
    print(f'  {k}: {np.round(v, 3)}')
print('\nCondicional vishing:')
for k, v in cond_table_vish.items():
    print(f'  {k}: {np.round(v, 3)}')

## 5. Mixed-Type Gaussian Copula Augmenter

**Cambio central**: la matriz de correlación se estima sobre normal scores (`Φ⁻¹(rank/(n+1))`) de **todas** las columnas del copula, incluidas integer y binarias. Al muestrear:

- Continuas → interpolación lineal en cuantiles empíricos + jitter aditivo proporcional al std local.
- Integer → inverse-CDF empírica vía `searchsorted` (respeta multimodalidad y colas).
- Binarias → threshold sobre uniforme correlacionado: `1 si u > (1-p) else 0`. Esto preserva la correlación tetracórica con las continuas y la co-ocurrencia entre binarias.

La matriz correlacional se proyecta a la PD-más-cercana vía clipping de autovalores.

In [ ]:
class MixedTypeCopulaAugmenter:
    """
    Copula gaussiano sobre features de tipo mixto (continuas + integer + binarias).

    Preserva:
    - Marginales empíricas de cada feature (via inverse transform / searchsorted / threshold).
    - Estructura de correlación conjunta sobre TODAS las features numéricas
      (via correlación en el espacio de normal scores + Cholesky).
    - Restricciones de dominio (aplicadas fuera de la clase).
    """

    def __init__(self, df_class, continuous_cols, integer_cols, binary_cols, jitter_ties=1e-4):
        self.continuous_cols = list(continuous_cols)
        self.integer_cols = list(integer_cols)
        self.binary_cols = list(binary_cols)
        self.all_cols = self.continuous_cols + self.integer_cols + self.binary_cols

        self.df_class = df_class[self.all_cols].copy().reset_index(drop=True)
        self.n_original = len(self.df_class)
        self.stds = self.df_class[self.continuous_cols].std()

        # --- Marginales continuas: valores ordenados + posiciones para np.interp ---
        self.sorted_continuous = {}
        self.q_positions = {}
        for col in self.continuous_cols:
            vals = np.sort(self.df_class[col].values)
            self.sorted_continuous[col] = vals
            self.q_positions[col] = np.linspace(0.0, 1.0, len(vals))

        # --- Marginales integer: CDF empírica ---
        self.integer_cdf = {}
        for col in self.integer_cols:
            vals = self.df_class[col].values
            unique, counts = np.unique(vals, return_counts=True)
            cum_probs = np.cumsum(counts) / counts.sum()
            self.integer_cdf[col] = (unique, cum_probs)

        # --- Marginales binarias: P(1) ---
        self.binary_probs = {col: float(self.df_class[col].mean()) for col in self.binary_cols}

        # --- Normal scores y correlación copula ---
        normal_scores = np.zeros((self.n_original, len(self.all_cols)))
        for i, col in enumerate(self.all_cols):
            x = self.df_class[col].values.astype(float)
            # Random jitter sobre ties (crucial para binarias e integer con ties fuertes)
            x_jitter = x + RNG.normal(0, jitter_ties, size=self.n_original)
            ranks = pd.Series(x_jitter).rank(method='average').values
            u = (ranks - 0.5) / self.n_original
            u = np.clip(u, 1e-6, 1 - 1e-6)
            normal_scores[:, i] = scipy_stats.norm.ppf(u)

        corr = np.corrcoef(normal_scores.T)
        self.corr_matrix = self._nearest_correlation_pd(corr)
        self.cholesky = np.linalg.cholesky(self.corr_matrix)

    @staticmethod
    def _nearest_correlation_pd(A, eps=1e-6):
        """Proyecta A a la matriz de correlación PD más cercana (clip de autovalores)."""
        A = (A + A.T) / 2.0
        eigvals, eigvecs = np.linalg.eigh(A)
        eigvals = np.maximum(eigvals, eps)
        A_pd = eigvecs @ np.diag(eigvals) @ eigvecs.T
        # Reescalar a diagonal 1 (matriz de correlación)
        d = np.sqrt(np.diag(A_pd))
        A_pd = A_pd / np.outer(d, d)
        return A_pd

    def generate(self, n_samples, noise_scale=0.03, seed=None):
        """
        Genera n_samples filas sintéticas.

        noise_scale: jitter aditivo sobre las marginales continuas, en unidades de std local.
        """
        rng = np.random.default_rng(seed) if seed is not None else RNG
        p = len(self.all_cols)

        # 1) Muestreo del copula gaussiano correlacionado
        z = rng.normal(0.0, 1.0, size=(n_samples, p))
        z_corr = z @ self.cholesky.T
        u = scipy_stats.norm.cdf(z_corr)                 # uniforms correlacionadas

        result = pd.DataFrame(index=range(n_samples))

        # 2) Continuas: interpolación lineal + jitter aditivo
        for i, col in enumerate(self.continuous_cols):
            sorted_vals = self.sorted_continuous[col]
            positions = self.q_positions[col]
            values = np.interp(u[:, i], positions, sorted_vals)
            local_scale = self.stds[col] * noise_scale
            if local_scale > 0:
                values = values + rng.normal(0.0, local_scale, size=n_samples)
            result[col] = values

        # 3) Integer: inverse-CDF empírica via searchsorted
        offset = len(self.continuous_cols)
        for j, col in enumerate(self.integer_cols):
            unique, cum_probs = self.integer_cdf[col]
            idx = np.searchsorted(cum_probs, u[:, offset + j], side='left')
            idx = np.clip(idx, 0, len(unique) - 1)
            result[col] = unique[idx]

        # 4) Binarias: threshold sobre la uniforme correlacionada
        offset = len(self.continuous_cols) + len(self.integer_cols)
        for k, col in enumerate(self.binary_cols):
            p_col = self.binary_probs[col]
            result[col] = (u[:, offset + k] > (1.0 - p_col)).astype(int)

        return result


print('✅ MixedTypeCopulaAugmenter defined')

## 6. Generación del dataset 1M

In [ ]:
N_TOTAL = 1_000_000
VISHING_PCT = 0.015

n_vishing_target = int(N_TOTAL * VISHING_PCT)
n_legit_target = N_TOTAL - n_vishing_target

df_orig_legit = df_mobile[df_mobile['is_vishing'] == 0].reset_index(drop=True)
df_orig_vishing = df_mobile[df_mobile['is_vishing'] == 1].reset_index(drop=True)

n_legit_to_generate = n_legit_target - len(df_orig_legit)
n_vishing_to_generate = n_vishing_target - len(df_orig_vishing)

print('GENERATION PLAN')
print('=' * 60)
print(f'Total target:            {N_TOTAL:>12,}')
print(f'Vishing target ({VISHING_PCT*100:.1f}%): {n_vishing_target:>12,}')
print(f'Legit target:            {n_legit_target:>12,}')
print(f'\nOriginal legit:          {len(df_orig_legit):>12,}')
print(f'Original vishing:        {len(df_orig_vishing):>12,}')
print(f'\nTo generate legit:       {n_legit_to_generate:>12,}')
print(f'To generate vishing:     {n_vishing_to_generate:>12,}')

In [ ]:
print('Building augmenter for LEGIT class...')
t0 = time.time()
aug_legit = MixedTypeCopulaAugmenter(df_orig_legit, continuous_cols, integer_cols, binary_cols)
print(f'  ✅ ({time.time()-t0:.1f}s) — corr matrix {aug_legit.corr_matrix.shape}')

print('\nBuilding augmenter for VISHING class...')
t0 = time.time()
aug_vishing = MixedTypeCopulaAugmenter(df_orig_vishing, continuous_cols, integer_cols, binary_cols)
print(f'  ✅ ({time.time()-t0:.1f}s) — corr matrix {aug_vishing.corr_matrix.shape}')

In [ ]:
print('Generating synthetic LEGIT samples...')
t0 = time.time()
df_synth_legit = aug_legit.generate(n_legit_to_generate, noise_scale=0.03, seed=101)
print(f'  ✅ {len(df_synth_legit):,} rows in {time.time()-t0:.1f}s')

print('\nGenerating synthetic VISHING samples...')
t0 = time.time()
df_synth_vishing = aug_vishing.generate(n_vishing_to_generate, noise_scale=0.03, seed=202)
print(f'  ✅ {len(df_synth_vishing):,} rows in {time.time()-t0:.1f}s')

## 7. Post-processing: constraints, reglas, derivadas y metadata

Aquí es donde se cierra el ciclo. Aplicamos, en orden:

1. Constraints de dominio (rangos, no-negatividad, tipos enteros).
2. Reglas determinísticas (`is_atypical_hour`, consistencias con `phone_call_active` / `transaction_attempted`).
3. Indicadores BioCatch por condicional empírica sobre `biocatch_risk_score`.
4. Recalculo de features derivadas (`errors_per_minute`, `interactions_per_s`, `hesitation_composite`).
5. Metadata (session_id, customer_id aleatorio, timestamp coherente con `hour_of_day`, os_type mobile).

In [ ]:
RATIO_COLS_01 = ['segmented_typing_ratio', 'avg_touch_pressure', 'dead_time_ratio',
                 'data_familiarity_score', 'keystroke_variability', 'swipe_directional_variance']

def enforce_domain_constraints(df, integer_cols, binary_cols, continuous_cols, mins_reference):
    # Ratios en [0, 1]
    for col in RATIO_COLS_01:
        if col in df.columns:
            df[col] = df[col].clip(0, 1)

    # No-negatividad para continuas originalmente positivas
    for col in continuous_cols:
        if col in df.columns and mins_reference.get(col, 0) >= 0:
            df[col] = df[col].clip(lower=0)

    # Integer -> redondeo, no-negativos, dtype int
    for col in integer_cols:
        if col in df.columns:
            df[col] = np.rint(df[col]).clip(lower=0).astype(int)

    # Binarias -> 0/1
    for col in binary_cols:
        if col in df.columns:
            df[col] = df[col].astype(int).clip(0, 1)

    # BioCatch scores en [0, 1000]
    for col in ['biocatch_risk_score', 'biocatch_genuine_score']:
        if col in df.columns:
            df[col] = df[col].clip(0, 1000).astype(int)

    # hour_of_day en [0, 23], tilt en [0, 90]
    if 'hour_of_day' in df.columns:
        df['hour_of_day'] = df['hour_of_day'].clip(0, 23).astype(int)
    if 'device_tilt_angle_mean' in df.columns:
        df['device_tilt_angle_mean'] = df['device_tilt_angle_mean'].clip(0, 90)
    return df


def enforce_logical_consistencies(df):
    # is_atypical_hour determinístico
    df['is_atypical_hour'] = compute_atypical_hour(df['hour_of_day']).astype(int)

    # unique_screens_visited <= screens_visited
    df['unique_screens_visited'] = np.minimum(df['unique_screens_visited'], df['screens_visited'])

    # Si no hay llamada, no hay overlap
    df.loc[df['phone_call_active'] == 0, 'call_overlap_duration_s'] = 0.0

    # Si no hay transacción, resetear campos ligados
    mask_no_tx = df['transaction_attempted'] == 0
    df.loc[mask_no_tx, 'time_to_transaction_s'] = 0.0
    df.loc[mask_no_tx, 'transaction_amount_cop'] = 0
    df.loc[mask_no_tx, 'is_new_beneficiary'] = 0

    # Consistencia interna hesitation
    df.loc[df['hesitation_count'] == 0, 'avg_hesitation_duration_s'] = 0.0
    df.loc[df['hesitation_count'] == 0, 'max_hesitation_duration_s'] = 0.0
    # max >= avg
    df['max_hesitation_duration_s'] = np.maximum(df['max_hesitation_duration_s'],
                                                 df['avg_hesitation_duration_s'])

    # dead time consistency
    df.loc[df['dead_time_periods'] == 0, 'total_dead_time_s'] = 0.0
    df.loc[df['dead_time_periods'] == 0, 'dead_time_ratio'] = 0.0
    return df


def apply_biocatch_indicators(df, edges, cond_table, seed=None):
    rng = np.random.default_rng(seed) if seed is not None else RNG
    bins = pd.cut(df['biocatch_risk_score'], bins=edges, labels=False, include_lowest=True)
    bins = bins.fillna(0).astype(int).clip(0, len(edges) - 2)
    for col, probs in cond_table.items():
        p_row = probs[bins.values]
        df[col] = (rng.random(len(df)) < p_row).astype(int)
    return df


def recompute_derived_features(df):
    dur = df['session_duration_s'].clip(lower=1.0)
    df['errors_per_minute'] = df['input_error_count'] / (dur / 60.0)
    # interactions_per_s: usamos screens_visited + input_error_count + input_correction_count
    # como proxy de "interactions" (coherente con el diccionario: interacciones por segundo del flujo).
    interactions = (df['screens_visited'] + df['input_error_count']
                    + df['input_correction_count'] + df['copy_paste_events'])
    df['interactions_per_s'] = interactions / dur
    df['hesitation_composite'] = (df['hesitation_count'] * df['avg_hesitation_duration_s']) / dur
    return df


print('✅ Post-processing helpers ready')

In [ ]:
from datetime import datetime, timedelta

def assemble_class(df_orig_class, df_synth_class, label, edges, cond_table,
                   continuous_cols, integer_cols, binary_cols):
    # Referencia de mínimos por columna (del original) para constraints
    mins_ref = df_orig_class[continuous_cols].min().to_dict()

    # Marca de sintético
    orig_block = df_orig_class[continuous_cols + integer_cols + binary_cols].copy()
    orig_block['is_synthetic'] = 0
    synth_block = df_synth_class.copy()
    synth_block['is_synthetic'] = 1

    combined = pd.concat([orig_block, synth_block], ignore_index=True)

    # Constraints -> reglas -> indicators -> derivadas
    combined = enforce_domain_constraints(combined, integer_cols, binary_cols,
                                           continuous_cols, mins_ref)
    combined = enforce_logical_consistencies(combined)
    combined = apply_biocatch_indicators(combined, edges, cond_table, seed=999)
    combined = recompute_derived_features(combined)

    combined['is_vishing'] = label
    return combined


print('Assembling LEGIT block...')
t0 = time.time()
block_legit = assemble_class(df_orig_legit, df_synth_legit, label=0,
                             edges=edges_legit, cond_table=cond_table_legit,
                             continuous_cols=continuous_cols,
                             integer_cols=integer_cols,
                             binary_cols=binary_cols)
print(f'  ✅ {len(block_legit):,} rows in {time.time()-t0:.1f}s')

print('Assembling VISHING block...')
t0 = time.time()
block_vish = assemble_class(df_orig_vishing, df_synth_vishing, label=1,
                            edges=edges_vish, cond_table=cond_table_vish,
                            continuous_cols=continuous_cols,
                            integer_cols=integer_cols,
                            binary_cols=binary_cols)
print(f'  ✅ {len(block_vish):,} rows in {time.time()-t0:.1f}s')

df_augmented = pd.concat([block_legit, block_vish], ignore_index=True)
df_augmented = df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'\nTotal augmented: {len(df_augmented):,}')
print(f'Vishing rate: {df_augmented["is_vishing"].mean()*100:.3f}%')

In [ ]:
# Metadata: session_id, customer_id, session_timestamp coherente con hour_of_day, os_type mobile
n = len(df_augmented)

df_augmented.insert(0, 'session_id', [f'SES-{i:07d}' for i in range(1, n + 1)])

# customer_id aleatorio (según instrucción del usuario, no modelamos baseline por cliente)
df_augmented.insert(1, 'customer_id',
                    [f'CUS-{RNG.integers(10000, 100000)}' for _ in range(n)])

# session_timestamp coherente con hour_of_day
base_date = datetime(2024, 6, 1)
days_offsets = RNG.integers(0, 365, size=n)
mins = RNG.integers(0, 60, size=n)
secs = RNG.integers(0, 60, size=n)
hours = df_augmented['hour_of_day'].values

timestamps = [base_date + timedelta(days=int(days_offsets[i]),
                                     hours=int(hours[i]),
                                     minutes=int(mins[i]),
                                     seconds=int(secs[i])) for i in range(n)]
df_augmented.insert(2, 'session_timestamp', timestamps)

# device_type = mobile (todo el augmentation es mobile)
df_augmented.insert(3, 'device_type', 'mobile')

# os_type respetando la distribución mobile original
os_probs = df_mobile['os_type'].value_counts(normalize=True)
df_augmented.insert(4, 'os_type',
                    RNG.choice(os_probs.index.values, size=n, p=os_probs.values))

# app_version respetando distribución original
app_probs = df_mobile['app_version'].value_counts(normalize=True)
df_augmented.insert(5, 'app_version',
                    RNG.choice(app_probs.index.values, size=n, p=app_probs.values))

# days_to_claim: bootstrap del pool de vishing original; -1 para legit
mask_v = df_augmented['is_vishing'] == 1
n_v = int(mask_v.sum())
df_augmented['days_to_claim'] = -1
df_augmented.loc[mask_v, 'days_to_claim'] = RNG.choice(days_to_claim_pool, size=n_v)

# claim_category: distribución empírica; 'none' para legit
df_augmented['claim_category'] = 'none'
df_augmented.loc[mask_v, 'claim_category'] = RNG.choice(
    claim_cat_probs.index.values, size=n_v, p=claim_cat_probs.values)

print(f'✅ Metadata attached — final shape: {df_augmented.shape}')
print(f'Columns: {list(df_augmented.columns)[:10]}...')
print(f'\nVishing rate final: {df_augmented["is_vishing"].mean()*100:.3f}%')
print(f'Synthetic rate:     {df_augmented["is_synthetic"].mean()*100:.2f}%')

## 8. Guardado en S3

In [ ]:
output_path = 's3://poc-fraude-vishing/data/augmented/dataset_1M_vishing_v2.parquet'

print(f'Saving to {output_path} ...')
t0 = time.time()
df_augmented.to_parquet(output_path, engine='pyarrow', index=False)
print(f'✅ Saved in {time.time()-t0:.1f}s')

## 9. Validación de fidelidad

Tres chequeos antes de entrenar sobre el 1M:

1. **Marginales** — KS test (continuas) y distancia de variación total (integer/binarias) por feature, comparando sintéticos vs. originales dentro de la misma clase.
2. **Correlación** — distancia de Frobenius normalizada entre la matriz de correlación original y la sintética, por clase.
3. **Distinguibilidad** — un XGBoost que intenta separar "original" de "sintético" dentro de la misma clase. AUC objetivo ~0.55–0.65. Si AUC > 0.75, hay que subir `noise_scale` o refinar marginales.

In [ ]:
def marginal_fidelity_report(df_orig, df_synth, continuous_cols, integer_cols, binary_cols):
    rows = []
    for col in continuous_cols:
        stat, p = scipy_stats.ks_2samp(df_orig[col], df_synth[col])
        rows.append({'feature': col, 'type': 'continuous', 'metric': 'KS_stat',
                     'value': stat, 'p_value': p})
    for col in integer_cols + binary_cols:
        # TV distance sobre las frecuencias de valores
        vc_o = df_orig[col].value_counts(normalize=True)
        vc_s = df_synth[col].value_counts(normalize=True)
        idx = vc_o.index.union(vc_s.index)
        tv = 0.5 * (vc_o.reindex(idx, fill_value=0) - vc_s.reindex(idx, fill_value=0)).abs().sum()
        rows.append({'feature': col,
                     'type': 'integer' if col in integer_cols else 'binary',
                     'metric': 'TV_dist',
                     'value': tv, 'p_value': np.nan})
    return pd.DataFrame(rows)

# Comparamos synthetic-only (is_synthetic=1) contra original (is_synthetic=0), por clase
report_legit = marginal_fidelity_report(
    df_augmented[(df_augmented['is_vishing']==0) & (df_augmented['is_synthetic']==0)],
    df_augmented[(df_augmented['is_vishing']==0) & (df_augmented['is_synthetic']==1)],
    continuous_cols, integer_cols, binary_cols)

report_vish = marginal_fidelity_report(
    df_augmented[(df_augmented['is_vishing']==1) & (df_augmented['is_synthetic']==0)],
    df_augmented[(df_augmented['is_vishing']==1) & (df_augmented['is_synthetic']==1)],
    continuous_cols, integer_cols, binary_cols)

print('=== TOP 10 features con peor fidelidad marginal — LEGIT ===')
print(report_legit.sort_values('value', ascending=False).head(10).to_string(index=False))
print('\n=== TOP 10 features con peor fidelidad marginal — VISHING ===')
print(report_vish.sort_values('value', ascending=False).head(10).to_string(index=False))

In [ ]:
def correlation_fidelity(df_orig, df_synth, cols):
    C_o = df_orig[cols].corr().values
    C_s = df_synth[cols].corr().values
    diff = C_o - C_s
    frob = np.linalg.norm(diff, 'fro')
    frob_norm = frob / np.linalg.norm(C_o, 'fro')
    max_abs_delta = np.max(np.abs(diff))
    return frob, frob_norm, max_abs_delta

cols_all = continuous_cols + integer_cols + binary_cols

frob_l, frob_l_n, max_l = correlation_fidelity(
    df_augmented[(df_augmented['is_vishing']==0) & (df_augmented['is_synthetic']==0)],
    df_augmented[(df_augmented['is_vishing']==0) & (df_augmented['is_synthetic']==1)],
    cols_all)

frob_v, frob_v_n, max_v = correlation_fidelity(
    df_augmented[(df_augmented['is_vishing']==1) & (df_augmented['is_synthetic']==0)],
    df_augmented[(df_augmented['is_vishing']==1) & (df_augmented['is_synthetic']==1)],
    cols_all)

print('=== Fidelidad de correlación (matriz completa) ===')
print(f'LEGIT   — Frobenius: {frob_l:8.3f}  |  Normalizada: {frob_l_n:.4f}  |  max|Δρ|: {max_l:.3f}')
print(f'VISHING — Frobenius: {frob_v:8.3f}  |  Normalizada: {frob_v_n:.4f}  |  max|Δρ|: {max_v:.3f}')
print('\nUmbrales sugeridos: normalizada < 0.15 (buena), max|Δρ| < 0.15 (buena).')

In [ ]:
# Distinguishability test: original vs synthetic dentro de la misma clase
try:
    import xgboost as xgb
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score

    def distinguishability_auc(df_class, cols, label='is_synthetic', sample_size=40000):
        # Balanceamos originales y sintéticos para el test
        pos = df_class[df_class[label] == 1]
        neg = df_class[df_class[label] == 0]
        n_take = min(len(pos), len(neg), sample_size // 2)
        sample = pd.concat([pos.sample(n_take, random_state=1), neg.sample(n_take, random_state=1)])
        X = sample[cols].values
        y = sample[label].values
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
        clf = xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                                 use_label_encoder=False, eval_metric='logloss',
                                 tree_method='hist', n_jobs=-1)
        clf.fit(X_tr, y_tr)
        proba = clf.predict_proba(X_te)[:, 1]
        return roc_auc_score(y_te, proba), clf

    auc_l, clf_l = distinguishability_auc(df_augmented[df_augmented['is_vishing']==0], cols_all)
    auc_v, clf_v = distinguishability_auc(df_augmented[df_augmented['is_vishing']==1], cols_all)

    print('=== Distinguishability AUC (original vs sintético) ===')
    print(f'LEGIT   — AUC: {auc_l:.4f}')
    print(f'VISHING — AUC: {auc_v:.4f}')
    print('\nInterpretación:')
    print('  ~0.50 = indistinguibles (ideal, pero también posible overfit del augmenter)')
    print('  0.55-0.65 = buena fidelidad, ligero patrón diferencial')
    print('  0.65-0.75 = fidelidad aceptable, revisar features con más importancia')
    print('  > 0.75  = sintéticos claramente separables → subir noise_scale o refinar')

    # Top features que "delatan" al sintético
    imp_l = pd.Series(clf_l.feature_importances_, index=cols_all).sort_values(ascending=False).head(10)
    imp_v = pd.Series(clf_v.feature_importances_, index=cols_all).sort_values(ascending=False).head(10)
    print('\nTop 10 features delatoras — LEGIT:')
    print(imp_l.to_string())
    print('\nTop 10 features delatoras — VISHING:')
    print(imp_v.to_string())

except ImportError:
    print('⚠️ xgboost no está instalado. Instálalo con: pip install xgboost')

## 10. Chequeo rápido de consistencias forzadas

Sanity check final: las reglas determinísticas deben cumplirse al 100%.

In [ ]:
checks = {
    'is_atypical_hour ↔ hour_of_day':
        (df_augmented['is_atypical_hour'] == df_augmented['hour_of_day'].isin(ATYPICAL_HOURS).astype(int)).mean(),
    'phone_call_active=0 ⇒ call_overlap_duration_s=0':
        ((df_augmented['phone_call_active']==1) | (df_augmented['call_overlap_duration_s']==0)).mean(),
    'transaction_attempted=0 ⇒ transaction_amount_cop=0':
        ((df_augmented['transaction_attempted']==1) | (df_augmented['transaction_amount_cop']==0)).mean(),
    'unique_screens_visited <= screens_visited':
        (df_augmented['unique_screens_visited'] <= df_augmented['screens_visited']).mean(),
    'max_hesitation_duration_s >= avg_hesitation_duration_s':
        (df_augmented['max_hesitation_duration_s'] >= df_augmented['avg_hesitation_duration_s']).mean(),
    'device_type == mobile':
        (df_augmented['device_type']=='mobile').mean(),
}
print('=== Consistencias forzadas (deben ser 1.000) ===')
for k, v in checks.items():
    ok = '✅' if v >= 0.9999 else '❌'
    print(f'  {ok} {k}: {v:.4f}')